# Compile with Vela, Profile on the Ethos-U85 NPU

This demo takes a plain INT8 LiteRT (TFLite) model, compiles it for the
Ethos-U85 NPU with Arm's **Vela** compiler using Ambiq's system
configuration, and profiles the result on the Atomiq110 FPGA with `hpx`.

> **Experimental FPGA target** — Atomiq110 support is best-effort and its
> measurements describe the current FPGA image only, not production
> silicon. See the *Atomiq110 NPU Profiling* example in the docs for the
> full caveats.

**Ethos-U in 30 seconds.** The NPU executes a *command stream* compiled
ahead of time. Vela replaces every subgraph the NPU supports with a single
`ethos-u` custom op containing that command stream; anything unsupported
stays on the Cortex-M55. Two inputs matter:

- `--accelerator-config` — the NPU variant, baked into the command stream.
  It **must match the target** (Atomiq110 = `ethos-u85-256`); HPX preflight
  rejects mismatches before touching hardware.
- `--config`/`--system-config`/`--memory-mode` — an `.ini` describing the
  system around the NPU (clocks, memory ports, latencies). The system
  config only shapes Vela's scheduling and static estimates. The memory
  mode decides which region holds weights and arena, so it must agree
  with the runtime's buffer placement and the driver's region setup.

**Prerequisites** — install both packages into this notebook's
environment, pinned for reproducibility. The `analysis` extra pulls in
`ai-edge-litert`, which HPX preflight needs to validate the model's Vela
accelerator config against the target NPU:

```bash
pip install "helia-profiler[analysis]" ethos-u-vela==4.5.0
```

In [ ]:
from pathlib import Path

import helia_profiler as hpx

try:
    from ethosu.vela import vela
except ImportError as exc:
    raise RuntimeError("Vela is missing — pip install ethos-u-vela==4.5.0") from exc

MODEL = hpx.examples.tiny_cnn()
VELA_INI = hpx.examples.ambiq_vela_ini()
OUT = Path.cwd() / "vela_out"
RESULTS = Path.cwd() / "results" / "vela_npu_demo"

BOARD = "atomiq110_fpga_turbo"
JLINK_SERIAL = ""  # set to disambiguate when several probes are attached
RUN_HARDWARE = True  # set False to read through the notebook without a board

print(f"hpx version: {hpx.__version__}")
print(f"Model:       {MODEL}")
print(f"Vela ini:    {VELA_INI}")

## 1. The Ambiq system configuration

`hpx.examples.ambiq_vela_ini()` materializes Ambiq's packaged Vela
configuration. Three system configs model the NPU clock tiers against the
fixed 250 MHz SRAM/AXI memory fabric:

| Section | NPU clock | Use |
|---|---|---|
| `Ambiq_ULP_SRAM` | 100 MHz | ultra-low-power operating point |
| `Ambiq_LP_SRAM` | 250 MHz | low-power operating point (**this demo**) |
| `Ambiq_HP_SRAM` | 500 MHz | high-performance operating point |

and two memory modes — this demo uses `Shared_Sram`: weights/consts in
read-only MRAM (Axi1), tensor arena in SRAM shared with the M55. That
matches the profile settings below (`weights_location="mram"`,
`arena_location="sram"`).

In [ ]:
print(VELA_INI.read_text()[:1100])

## 2. Compile the model

The input is HPX's packaged INT8 tiny CNN. Vela writes the compiled model
plus a summary of its static estimates.

Estimates are computed from the ini's system model at the LP tier's
250 MHz — the FPGA actually runs at 25 MHz, so treat them as *relative*
guidance and the profiler as the source of truth.

In [ ]:
vela.main(
    [
        "--accelerator-config",
        "ethos-u85-256",
        "--config",
        str(VELA_INI),
        "--system-config",
        "Ambiq_LP_SRAM",
        "--memory-mode",
        "Shared_Sram",
        "--output-dir",
        str(OUT),
        str(MODEL),
    ]
)

### What changed?

The compiled model should contain a single `CUSTOM (ethos-u)` op — the
whole graph fit on the NPU. (If parts of a model are unsupported, they
remain as CPU ops alongside one or more `ethos-u` ops.)

In [ ]:
VELA_MODEL = OUT / f"{MODEL.stem}_vela.tflite"
assert VELA_MODEL.exists(), "Vela did not produce the expected output"
print(f"{VELA_MODEL.name}: {VELA_MODEL.stat().st_size:,} bytes")
print(f"original:  {MODEL.stat().st_size:,} bytes ({MODEL.name})")

## 3. Profile on the FPGA

Connect the Atomiq110 FPGA board via J-Link and build the profiling
session: heliaRT with the `ethos_u` backend, weights in MRAM, arena in
SRAM, RTT transport (lossless).

If several probes are attached, find the right serial with
`session.probes()` (or `hpx probes match --board atomiq110_fpga_turbo` on
the command line) and set `JLINK_SERIAL` in the first cell.

In [ ]:
session = (
    hpx.Session()
    .with_model(VELA_MODEL, arena_size=524_288, arena_location="sram", weights_location="mram")
    .with_engine("helia-rt", backend="ethos_u")
    .with_target(board=BOARD, toolchain="gcc", transport="rtt")
    .with_profiling(
        iterations=5,
        warmup=2,
        per_layer=True,
        pmu_counters={"cpu": "default", "ethos_npu": "default"},
    )
    .with_output(dir=RESULTS)
)
if JLINK_SERIAL:
    session = session.with_target(jlink_serial=JLINK_SERIAL)

resolved = session.resolve()
print(f"engine:  {resolved.engine.type.value} (backend={resolved.engine.backend})")
print(f"board:   {resolved.target.board}")

In [ ]:
result = None
if RUN_HARDWARE:
    result = session.profile()
    print(f"{result.layer_count} layers, {result.total_cycles:,.0f} total CPU cycles")
else:
    print("Profile acquisition skipped. Set RUN_HARDWARE = True to build and flash.")

## 4. Read the results

The run captures two counter families per layer:

- `ARM_PMU_*` — the Cortex-M55's view (cycles, instructions, stalls)
- `ETHOSU_PMU_*` — the NPU's own counters for the `ethos-u` dispatch,
  plus `NPU_DISPATCHED` marking layers that actually ran on the NPU

Quick reads: `NPU_ACTIVE / CYCLE` is NPU utilization of the dispatch;
`MAC_ACTIVE / NPU_ACTIVE` separates compute-bound from memory-bound.
Remember the fused graph is **one** layer at runtime — for intra-graph
breakdowns use Vela's `--verbose-performance` static estimates.

In [ ]:
if result is not None:
    for layer in result.layers:
        npu = layer.counters.get("NPU_DISPATCHED")
        cyc = layer.counters.get("ETHOSU_PMU_CYCLE")
        act = layer.counters.get("ETHOSU_PMU_NPU_ACTIVE")
        mac = layer.counters.get("ETHOSU_PMU_MAC_ACTIVE")
        print(f"layer {layer.id!s:>3} {layer.op:<12} npu={npu} cycles={cyc} active={act} mac={mac}")
    dispatched = [l for l in result.layers if l.counters.get("NPU_DISPATCHED")]
    print(f"\n{len(dispatched)}/{result.layer_count} layers ran on the NPU")

## Where to go deeper

- *Compile with Vela and Profile on the NPU* (docs/examples) — this flow
  as shell commands
- *Atomiq110 NPU Profiling* (docs/examples) — counter presets, heliaAOT
  variant, FPGA caveats
- *PMU Counters* guide — the full `ethos_npu` counter catalogue